---
title: "Chapter 14: Time series"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/14-time-series.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/14-time-series.ipynb)

The attendance report says 87% for Monday. Is that better or worse than last month? You have the numbers but the date column is plain text -- pandas can't slice by month, compute week-over-week changes, or resample to monthly totals until the column is a proper datetime type with a DatetimeIndex. This notebook adds that infrastructure, using daily attendance records across five schools over a full term.

Part 10 (`13-combining-reshaping.ipynb`) introduced time series data and basic date parsing. This notebook goes deeper: DatetimeIndex slicing, `resample`, timezones, and lag features -- the same building blocks used in Part 12 (`15-polars.ipynb`) and, later, in forecasting.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

In [ ]:
import numpy as np
import pandas as pd

attendance = pd.read_csv("data/daily_attendance.csv")
attendance.dtypes

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 11 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Create and inspect `Timestamp` values, and parse dates out of text with `to_datetime` | Sec. 1 |
| 2 | Build a `DatetimeIndex` and use it to slice a time series by date | Sec. 2 |
| 3 | Select rows with a partial date string or a date range | Sec. 3 |
| 4 | Change the time granularity of a series with `resample` | Sec. 4 |
| 5 | Localize naive timestamps, convert between timezones, and store in UTC | Sec. 5 |
| 6 | Create lag and lead features with `shift`, measure autocorrelation | Sec. 6 |
:::


## 1. Date and Time Data Types

The `date` column above read in as plain text, `str` dtype, not a date pandas can do arithmetic on. `pd.to_datetime` converts it to pandas' dedicated datetime dtype, `datetime64`:

In [ ]:
attendance["date"] = pd.to_datetime(attendance["date"])
attendance.dtypes

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Pandas 3 infers the resolution it needs, not always nanoseconds</span><br><br>
Earlier pandas versions always stored datetimes as <code>datetime64[ns]</code>, nanosecond precision, whether the data needed it or not. Pandas 3's <code>to_datetime</code> infers a resolution from what is actually in the data: day-level strings like the ones here become <code>datetime64[s]</code> or coarser, not nanoseconds. <code>attendance["date"].dtype</code> shows whichever resolution was inferred for this column.
</div>

A single value out of a datetime column is a `Timestamp`, pandas' equivalent of Python's `datetime.datetime`, with the same year, month, day, and weekday attributes:

In [ ]:
first_day = attendance["date"].iloc[0]
print(type(first_day))
print(f"year={first_day.year}, month={first_day.month}, day_name={first_day.day_name()}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Parse and Inspect</span><br><br>

<b>Goal:</b> Convert a list of three date strings, <code>["2025-01-06", "2025-02-14", "2025-03-28"]</code>, to <code>Timestamp</code> values with <code>pd.to_datetime</code>, then print the day name of each one.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>dates = pd.to_datetime(["2025-01-06", "2025-02-14", "2025-03-28"])
for d in dates:
    print(d.day_name())</pre>
</div>

In [ ]:
# TODO: convert the three date strings and print each day name
...

## 2. The `DatetimeIndex`

Setting a datetime column as the index turns it into a `DatetimeIndex`, which unlocks date-based slicing instead of only position-based or exact-label lookups. Each school's rows are pulled out first, since a `DatetimeIndex` only makes sense for one time series at a time:

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Setting a datetime column as the index creates a `DatetimeIndex` that unlocks date-based slicing</span><br><br>
Once you call `.set_index('date')` on a datetime column, the result has a `DatetimeIndex`. You can then use partial date strings in `.loc` -- `school.loc['2025-02']` selects every row in February without spelling out start and end times. Without a `DatetimeIndex`, `.loc` requires an exact label match; with one, it understands periods.
</div>

In [ ]:
school_300 = attendance[attendance["school_id"] == 300].set_index("date")
school_300.index

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Setting the index before filtering to one entity</span><br><br>
<code>attendance.set_index("date")</code> on the full table produces a <code>DatetimeIndex</code> with the same date repeated once per school, since every school has a row for every day. Slicing that index by date then returns rows from every school mixed together for that date, not a clean single time series. Filter to one entity first, exactly as <code>school_300</code> does above, then set the index.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Build Another School's Series</span><br><br>

<b>Goal:</b> Filter <code>attendance</code> to <code>school_id == 302</code>, set <code>date</code> as the index, and confirm the result's index is a <code>DatetimeIndex</code> with <code>isinstance(result.index, pd.DatetimeIndex)</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>school_302 = attendance[attendance["school_id"] == 302].set_index("date")
isinstance(school_302.index, pd.DatetimeIndex)</pre>
</div>

In [ ]:
# TODO: filter to school_id 302, set date as index, confirm DatetimeIndex
...

## 3. Selecting Data from a Time Series

A `DatetimeIndex` accepts a partial date string in `.loc`, matching every row that falls inside it. `"2025-02"` selects the whole month without spelling out the first and last day:

In [ ]:
school_300.loc["2025-02"].head()

A slice with two partial dates selects everything between them, inclusive of both ends:

In [ ]:
school_300.loc["2025-02-01":"2025-02-07"]

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: Comparing the size of two date ranges</span><br><br>
<code>len(school_300.loc["2025-01"])</code> against <code>len(school_300.loc["2025-02"])</code> confirms the row count for each month matches its number of business days, the same `bdate_range` weekday-only pattern used to build this dataset in the first place.
</div>

In [ ]:
print(f"January business days  : {len(school_300.loc['2025-01'])}")
print(f"February business days : {len(school_300.loc['2025-02'])}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Filter the Last Two Weeks of Term</span><br><br>

<b>Goal:</b> Select every row in <code>school_300</code> from <code>"2025-03-15"</code> to <code>"2025-03-28"</code> inclusive, and print the mean <code>attendance_rate</code> over that range.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>last_two_weeks = school_300.loc["2025-03-15":"2025-03-28"]
last_two_weeks["attendance_rate"].mean()</pre>
</div>

In [ ]:
# TODO: select 2025-03-15 through 2025-03-28 and print the mean attendance_rate
...

## 4. The Power of Pandas: `resample`

`resample` changes the time granularity of a series: daily data summarized into weekly or monthly figures, the same split-apply-combine idea from Part 3's `groupby`, except the groups are time intervals instead of category values:

In [ ]:
weekly_attendance = school_300["attendance_rate"].resample("W").mean()
weekly_attendance.head()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>resample</code> groups by time interval, <code>groupby</code> groups by value</span><br><br>
<code>df.groupby("caste")</code> (Part 3) splits rows by whatever value is already in the <code>caste</code> column. <code>series.resample("W")</code> splits rows by which week their <code>DatetimeIndex</code> label falls into, intervals that did not exist as a column at all until <code>resample</code> created them. Both still end with an aggregation like <code>.mean()</code> to combine each group into one number.
</div>

Monthly resampling on the same series needs only a different frequency string. In pandas 3 the old `"M"` alias was removed; the replacement is `"ME"` (month-end), which anchors each bucket to the last calendar day of the month:

In [ ]:
monthly_attendance = school_300["attendance_rate"].resample("ME").mean()
monthly_attendance

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Resampling the whole DataFrame keeps every entity separate, with care</span><br><br>
<code>attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("W").mean()</code> resamples each school's series independently in one call, instead of looping over schools and resampling each one by hand. <code>groupby</code> before <code>resample</code> is what keeps the schools from being averaged together.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Monthly Comparison Across Schools</span><br><br>

<b>Goal:</b> Set <code>date</code> as the index on the full <code>attendance</code> table, group by <code>school_id</code>, and resample to monthly means in one chained call. Use <code>"ME"</code> (month-end): the pandas 3 replacement for the removed <code>"M"</code> alias.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("ME").mean()</pre>
</div>

In [ ]:
# TODO: set date as index, group by school_id, resample monthly, take the mean
...

## 5. Timezones

The timestamps in this dataset are naive: they carry no timezone information. That is fine for a single-country attendance dataset. It is not fine for anything that crosses system boundaries: API responses, cloud server logs, or sensor readings from devices in different cities. Naive timestamps from different sources silently offset against each other when you join them, with no error to warn you.

Two operations handle timezones in pandas:

- `tz_localize(tz)`: stamps a naive series with a timezone. The values don't change, only their meaning.
- `tz_convert(tz)`: shifts a timezone-aware series to a different timezone. The values change to represent the same instant in the target zone.

The standard practice for storage: localize to UTC, convert to local time only for display.

In [ ]:
# Localize the naive DatetimeIndex to East Africa Time (UTC+3, Tanzania/Kenya)
school_300_eat = school_300.copy()
school_300_eat.index = school_300_eat.index.tz_localize("Africa/Nairobi")
print("Localized (EAT):", school_300_eat.index[:2])

# Convert to UTC for storage
school_300_utc = school_300_eat.copy()
school_300_utc.index = school_300_utc.index.tz_convert("UTC")
print("UTC:            ", school_300_utc.index[:2])

![Timezone handling flow: a tz-naive timestamp from a CSV is localized to its source timezone, converted to UTC for storage, then converted to a local timezone only when displaying output. Mixing naive and aware timestamps in the same column raises a TypeError.](figs/timezone-flow.svg){fig-alt="Four boxes connected by arrows: tz-naive (dashed, gray), tz-aware (blue), UTC stored (green), local display (purple). A red warning box below warns against mixing naive and aware timestamps. A green tip box recommends storing in UTC."}

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Store in UTC, display in local time</span><br><br>
UTC has no daylight saving transitions and no ambiguous hours, so it's the only safe format for storing timestamps that will be compared, sorted, or joined across sources. Localize to UTC at ingestion; convert to the user's timezone only when formatting for display. Timestamps stored as naive local time break silently when DST shifts: two records that look 60 minutes apart may actually be 120 minutes apart, or the same instant twice.
</div>

Mixing naive and aware timestamps raises a `TypeError`, which is actually helpful: it prevents silent wrong answers.

```python
school_300_utc.index[0] > school_300.index[0]
# TypeError: Cannot compare tz-naive and tz-aware datetime-like objects
```

The fix is always to localize the naive series before comparing:

In [ ]:
# Compare two timezone-aware timestamps safely
school_300_eat_end = school_300_eat.index[-1]
school_300_utc_end = school_300_utc.index[-1]

# Same instant, different representation
print("EAT end:", school_300_eat_end)
print("UTC end:", school_300_utc_end)
print("Same instant:", school_300_eat_end == school_300_utc_end)

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Localizing when you should be converting</span><br><br>
<code>tz_localize</code> stamps the existing values with a timezone label: <code>2025-01-06 00:00</code> becomes <code>2025-01-06 00:00+03:00</code>. The number did not change. <code>tz_convert</code> shifts the values to represent the same instant elsewhere: <code>2025-01-06 00:00+03:00</code> becomes <code>2025-01-05 21:00+00:00</code>. If you call <code>tz_localize("UTC")</code> on data that is actually in EAT, you have mislabelled it. The timestamps will compare as if they are 3 hours earlier than they really are.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Localize and Convert</span><br><br>

<b>Goal:</b> Starting from <code>school_302</code> (built in Activity 2), localize its naive index to <code>"Africa/Nairobi"</code>, then convert to <code>"Europe/London"</code>. Print the first timestamp in both representations and confirm they are the same instant.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>school_302 = attendance[attendance["school_id"] == 302].set_index("date")
s302_eat = school_302.copy()
s302_eat.index = s302_eat.index.tz_localize("Africa/Nairobi")
s302_london = s302_eat.copy()
s302_london.index = s302_london.index.tz_convert("Europe/London")
print(s302_eat.index[0])
print(s302_london.index[0])</pre>
</div>

In [ ]:
# TODO: localize school_302 to Africa/Nairobi then convert to Europe/London
...

## 6. Lag, Lead, and Autocorrelation

A lag feature answers the question: "what was the value yesterday?" It is the most common feature engineering step for time series in ML. If Monday's attendance rate predicts Tuesday's, a lag-1 feature captures that relationship in a column a model can consume.

`shift(n)` moves values forward by `n` periods (positive = lag, negative = lead). The first `n` rows get `NaN` because there is no previous value to fill them.

In [ ]:
rate = school_300["attendance_rate"].copy()

# lag-1: yesterday's attendance rate
lag1 = rate.shift(1)
lag1.name = "rate_lag1"

# lag-5: one school week ago
lag5 = rate.shift(5)
lag5.name = "rate_lag5"

features = pd.concat([rate, lag1, lag5], axis=1)
features.head(8)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Lag features turn time series forecasting into supervised learning</span><br><br>
A gradient boosting model doesn't know what "time" means. It knows column values. Giving it a <code>rate_lag1</code> column is how you communicate "yesterday's value" in a language the model understands. A model trained on <code>[rate_lag1, rate_lag5, day_of_week]</code> predicting <code>attendance_rate</code> is a supervised regression problem built from a single time series. The NaN rows produced by <code>shift</code> must be dropped or filled before training.
</div>

In [ ]:
# Rolling 5-day average: smooth out day-of-week noise
rolling_mean = rate.rolling(window=5).mean()
rolling_mean.name = "rate_rolling5"

# Rolling 5-day std: flags volatile weeks
rolling_std = rate.rolling(window=5).std()
rolling_std.name = "rate_rolling5_std"

feature_matrix = pd.concat([rate, lag1, lag5, rolling_mean, rolling_std], axis=1)
feature_matrix.dropna().head()  # drop NaN rows before modelling

`autocorr(lag)` measures how strongly a series correlates with a delayed copy of itself. An autocorrelation close to 1 means yesterday's value is a strong predictor of today's: a useful sanity check before building a lag-based model.

In [ ]:
for lag in [1, 5, 10]:
    ac = rate.autocorr(lag=lag)
    print(f"autocorr(lag={lag:>2}): {ac:.3f}")

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: Reading autocorrelation results</span><br><br>
A lag-1 autocorrelation of ~0.4 to 0.7 is typical for daily attendance data: yesterday's rate is a moderate predictor of today's but not a perfect one. A value near 0 means the series is essentially random from day to day. A value near 1 means it barely changes at all. The lag-5 value reflects the weekly pattern: a Monday is most similar to the previous Monday, not to last Friday.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 6 - Build a Lag Feature Matrix</span><br><br>

<b>Goal:</b> For <code>school_302</code>, create a feature matrix with columns: the original <code>attendance_rate</code>, a lag-1 column, a lag-5 column, and a 3-day rolling mean. Drop NaN rows, then print the autocorrelation at lag 1 and lag 5. What do the values tell you about how predictable attendance is at school 302?
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>rate_302 = attendance[attendance["school_id"]==302].set_index("date")["attendance_rate"]
fm_302 = pd.concat([
    rate_302,
    rate_302.shift(1).rename("lag1"),
    rate_302.shift(5).rename("lag5"),
    rate_302.rolling(3).mean().rename("rolling3"),
], axis=1).dropna()
print(fm_302.head())
print("autocorr lag1:", rate_302.autocorr(lag=1).round(3))
print("autocorr lag5:", rate_302.autocorr(lag=5).round(3))</pre>
</div>

In [ ]:
# TODO: build lag feature matrix for school_302 and check autocorrelations
...

## Capstone: Term-End Attendance Report

Combine every operation from this notebook: parsing dates, building a per-school time series, slicing by date, and resampling, into one short report comparing the start and end of term.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Term-End Attendance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Set <code>date</code> as the index on the full <code>attendance</code> table (Sec. 2)</li>
<li>Group by <code>school_id</code> and resample to weekly means (Sec. 4)</li>
<li>From the result, select the first week of January and the last week of March for every school, using a partial date string (Sec. 3)</li>
<li>Report which school had the largest drop in attendance between those two weeks</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>weekly = attendance.set_index("date").groupby("school_id")["attendance_rate"].resample("W").mean()

first_week = weekly.loc[:, "2025-01-06":"2025-01-12"]
last_week = weekly.loc[:, "2025-03-24":"2025-03-28"]
drop_per_school = first_week.groupby("school_id").mean() - last_week.groupby("school_id").mean()
drop_per_school.sort_values(ascending=False)</pre>
</div>

In [ ]:
# TODO: build the term-end attendance report described above
...

## Further Reading

| Resource | Why it matters |
|---|---|
| McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly. | Chapter 11 (time series) is the most complete treatment of `DatetimeIndex`, `resample`, and `rolling` with pandas |
| [pandas documentation: Time series / date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html) | The authoritative reference for `pd.date_range`, offset aliases, time zones, and `Period` |
| [pandas documentation: Time zone handling](https://pandas.pydata.org/docs/user_guide/timeseries.html#time-zone-handling) | `tz_localize`, `tz_convert`, and the ambiguous-times edge cases |
| McKinney, W. (2011). [Time series analysis in Python with pandas](https://conference.scipy.org/proceedings/scipy2011/pdfs/wes_mckinney.pdf). *SciPy 2011*. | The original paper describing pandas' time series design; short and still accurate |
| Hyndman, R.J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice*, 3rd ed. | Free at [otexts.com/fpp3](https://otexts.com/fpp3): the next step after understanding how to *store* time series data is learning how to *model* it |

## Summary

| Concept | Key rule |
|---|---|
| `pd.to_datetime` | Parses text into pandas' `datetime64` dtype; pandas 3 infers the resolution instead of always using nanoseconds |
| `Timestamp` | A single datetime value, with `.year`, `.month`, `.day_name()`, and similar attributes |
| `DatetimeIndex` | Set a datetime column as the index to unlock date-based slicing |
| Filter before indexing | Set a `DatetimeIndex` on one entity's rows, not a table mixing several entities at the same dates |
| `.loc["2025-02"]` | A partial date string selects every row inside that period |
| `.loc[start:end]` | A date range slice is inclusive of both ends |
| `.resample("ME")` | Groups rows by month-end interval; `"ME"` is the pandas 3 replacement for the removed `"M"` alias |
| `groupby(...).resample(...)` | Resample each entity's series independently in one chained call |
| `tz_localize(tz)` | Stamps a naive timestamp with a timezone; values don't shift |
| `tz_convert(tz)` | Shifts a timezone-aware timestamp to a different zone; values change |
| Store UTC | Localize to UTC at ingestion, convert to local time only for display |
| `shift(n)` | Creates lag (positive n) or lead (negative n) features; first n rows become NaN |
| `rolling(n).mean()` | Rolling window average; smooths noise and is a useful feature in its own right |
| `autocorr(lag=n)` | Measures how strongly a series predicts itself n steps ahead; guides lag feature selection |

**Next:** `15-polars.ipynb`, covering Polars' DataFrame, expression API, lazy evaluation, and streaming.